In [1]:
# STEP 2 - one-time, read-only check: event week vs quiet week from the January files.
# Reads two files and their sidecars. Writes nothing, creates nothing, uploads nothing.

import hashlib
import json
from pathlib import Path

import pandas as pd

from resolveiq.config import REPO_ROOT

EVENT = "event_2024-01-15_22"    # created 2024-01-15 (inclusive) up to 2024-01-23 (exclusive)
QUIET = "quiet_2024-01-24_30"    # created 2024-01-24 (inclusive) up to 2024-01-31 (exclusive)

FEATURES_PATH = REPO_ROOT / "data" / "labelled" / "heat_jan2024_event_quiet_features_at_intake.json"
LABELS_PATH = REPO_ROOT / "data" / "labelled" / "heat_jan2024_event_quiet_slow_resolution_labels.json"
WATCHED_DIR = FEATURES_PATH.parent

listing_before = sorted(p.name for p in WATCHED_DIR.iterdir())


def sha256_of(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def load_and_verify(path):
    sidecar = json.loads(path.with_suffix(".meta.json").read_text(encoding="utf-8"))
    digest = sha256_of(path)
    size = path.stat().st_size
    print(path.name)
    print(f"    on disk  {size:>10,} bytes   sha256 {digest[:16]}...")
    print(f"    sidecar  {sidecar['artifact_bytes']:>10,} bytes   "
          f"sha256 {sidecar['artifact_sha256'][:16]}...")
    ok = digest == sidecar["artifact_sha256"] and size == sidecar["artifact_bytes"]
    print(f"    -> {'MATCH' if ok else 'DIFFERENT - STOP'}")
    assert digest == sidecar["artifact_sha256"], f"{path.name}: not the bytes the sidecar hashed"
    assert size == sidecar["artifact_bytes"], f"{path.name}: the size moved"

    frame = pd.DataFrame(json.loads(path.read_text(encoding="utf-8")))
    frame["unique_key"] = frame["unique_key"].astype("int64")
    assert frame["unique_key"].is_unique, f"{path.name}: duplicate unique_key"
    assert len(frame) == int(sidecar["rows"]), f"{path.name}: row count vs sidecar"
    print(f"    -> loaded {len(frame):,} rows x {len(frame.columns)} columns")
    return frame, sidecar


print("1. integrity - the rest of this cell is only trustworthy if both lines say MATCH\n")
features, features_sidecar = load_and_verify(FEATURES_PATH)
labels, labels_sidecar = load_and_verify(LABELS_PATH)

merged = labels.merge(features, on="unique_key", how="inner", validate="one_to_one")
assert len(merged) == len(labels), f"only {len(merged):,} of {len(labels):,} matched"

thresholds = merged["label_threshold_hours"].unique()
assert len(thresholds) == 1, f"more than one project threshold in this file: {list(thresholds)}"
project_threshold_hours = float(thresholds[0])

arrived = features.groupby("window")["unique_key"].count()
closed = merged.groupby("window")["unique_key"].count()
slow = merged.assign(_slow=merged["slow_resolution"].astype("int64")).groupby("window")["_slow"].sum()
still_open = arrived - closed
slow_share_pct = 100.0 * slow / closed

print("\n2. the comparison, per window\n")
print(f"{'':39}{'event week':>16}{'quiet week':>16}")
for label, series, cast, spec in (
    ("arrived (every created ticket)", arrived, int, "{:>16,}"),
    ("closed", closed, int, "{:>16,}"),
    ("still open - carries no label", still_open, int, "{:>16,}"),
    ("closed later than project threshold", slow, int, "{:>16,}"),
    ("share of CLOSED left too long", slow_share_pct, float, "{:>15,.1f}%"),
):
    print(f"{label:39}{spec.format(cast(series[EVENT]))}{spec.format(cast(series[QUIET]))}")

print("\n3. pooled - the two windows added together, computed once from totals\n")
arrived_total = int(arrived.sum())
closed_total = int(closed.sum())
still_open_total = arrived_total - closed_total
slow_total = int(slow.sum())
pooled_share_pct = 100.0 * slow_total / closed_total
print(f"    arrived {arrived_total:,}   closed {closed_total:,}   still open {still_open_total:,}")
print(f"    slow {slow_total:,} of {closed_total:,} closed = {pooled_share_pct:.1f}% among observed closures")
print("    (pooled is slow_total / closed_total - not the average of the two window shares)")

print("\n4. cross-checks against what the sidecars already recorded\n")
checks = []
for window in (EVENT, QUIET):
    pulled = int(features_sidecar["windows"][window]["rows_pulled"])
    checks.append((f"arrived {window}", int(arrived[window]) == pulled,
                   f"{int(arrived[window]):,} vs sidecar {pulled:,}"))
checks.append(("arrived total vs features sidecar rows",
               arrived_total == int(features_sidecar["rows"]),
               f"{arrived_total:,} vs {int(features_sidecar['rows']):,}"))
checks.append(("closed total vs labels sidecar rows",
               closed_total == int(labels_sidecar["rows"]),
               f"{closed_total:,} vs {int(labels_sidecar['rows']):,}"))
checks.append(("still open vs labels sidecar open_tickets.count",
               still_open_total == int(labels_sidecar["open_tickets"]["count"]),
               f"{still_open_total:,} vs {int(labels_sidecar['open_tickets']['count']):,}"))
for short, recorded in labels_sidecar["slow_by_window_pct"].items():
    window = EVENT if short == "event" else QUIET
    checks.append((f"slow share {short} window",
                   abs(float(slow_share_pct[window]) - float(recorded)) < 0.05,
                   f"{float(slow_share_pct[window]):.1f}% vs {float(recorded):.1f}%"))
checks.append(("pooled slow share",
               abs(pooled_share_pct - float(labels_sidecar["slow_overall_pct"])) < 0.05,
               f"{pooled_share_pct:.1f}% vs {float(labels_sidecar['slow_overall_pct']):.1f}%"))
checks.append(("project threshold vs sidecar",
               abs(project_threshold_hours - float(labels_sidecar["label_threshold_hours"])) < 1e-9,
               f"{project_threshold_hours!r} h"))
outcome_columns = sorted(set(features.columns) & {
    "status", "closed_date", "resolution_description",
    "resolution_action_updated_date", "due_date"})
checks.append(("features file carries no outcome columns", not outcome_columns, str(outcome_columns)))

for name, ok, detail in checks:
    print(f"    {'OK  ' if ok else 'FAIL'} {name:46} {detail}")
failed = [name for name, ok, _ in checks if not ok]
assert not failed, f"these disagree with the sidecars: {failed}"

print(f"\n    the sidecar calls the threshold: {labels_sidecar['threshold_source']}")
print(f"    fitted on: {labels_sidecar['threshold_fitted_on']}")
print(f"    about open tickets: {labels_sidecar['open_tickets']['label']}")

print("\n5. read-only proof\n")
listing_after = sorted(p.name for p in WATCHED_DIR.iterdir())
assert listing_after == listing_before, (
    f"files changed inside {WATCHED_DIR.name}: {set(listing_after) ^ set(listing_before)}")
print(f"    {WATCHED_DIR.name}/ still lists the same {len(listing_after)} entries -> "
      f"these figures came from the same bytes Notebook 03 wrote, and nothing was added")

1. integrity - the rest of this cell is only trustworthy if both lines say MATCH

heat_jan2024_event_quiet_features_at_intake.json
    on disk  14,221,430 bytes   sha256 8d26ccbb40f73d14...
    sidecar  14,221,430 bytes   sha256 8d26ccbb40f73d14...
    -> MATCH
    -> loaded 30,126 rows x 13 columns
heat_jan2024_event_quiet_slow_resolution_labels.json
    on disk   3,632,305 bytes   sha256 67caa358422303cc...
    sidecar   3,632,305 bytes   sha256 67caa358422303cc...
    -> MATCH
    -> loaded 30,079 rows x 3 columns

2. the comparison, per window

                                             event week      quiet week
arrived (every created ticket)                   22,811           7,315
closed                                           22,770           7,309
still open - carries no label                        41               6
closed later than project threshold               6,526             731
share of CLOSED left too long                     28.7%           10.0%

3. pooled - 

In [2]:
print(f"unweighted mean of the two window shares: {(28.7 + 10.0) / 2:.2f}%")
print(f"weighted (7,257 / 30,079):               {100.0 * 7257 / 30079:.2f}%")
print("the pooled number is dominated by the 22,770-ticket window, which is correct")

unweighted mean of the two window shares: 19.35%
weighted (7,257 / 30,079):               24.13%
the pooled number is dominated by the 22,770-ticket window, which is correct


In [3]:
# STEP 3a - the verified calculation as one pure, JSON-ready function. Read-only.
import hashlib
import json
from datetime import date
from pathlib import Path

import pandas as pd

from resolveiq.config import REPO_ROOT

DATA_DIR = REPO_ROOT / "data" / "labelled"
FEATURES_PATH = DATA_DIR / "heat_jan2024_event_quiet_features_at_intake.json"
LABELS_PATH = DATA_DIR / "heat_jan2024_event_quiet_slow_resolution_labels.json"

EVENT = "event_2024-01-15_22"
QUIET = "quiet_2024-01-24_30"
OUTCOME_COLUMNS = {"status", "closed_date", "resolution_description",
                   "resolution_action_updated_date", "due_date"}


def _read_verified(path: Path):
    """Load one January artefact only if its bytes still match its own sidecar."""
    sidecar = json.loads(path.with_suffix(".meta.json").read_text(encoding="utf-8"))
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    assert digest == sidecar["artifact_sha256"], f"{path.name}: not the bytes the sidecar hashed"
    assert path.stat().st_size == int(sidecar["artifact_bytes"]), f"{path.name}: the size moved"
    frame = pd.DataFrame(json.loads(path.read_text(encoding="utf-8")))
    frame["unique_key"] = frame["unique_key"].astype("int64")
    assert frame["unique_key"].is_unique, f"{path.name}: duplicate unique_key"
    assert len(frame) == int(sidecar["rows"]), f"{path.name}: row count vs sidecar"
    provenance = {
        "file": str(path.relative_to(REPO_ROOT)).replace("\\", "/"),
        "sha256": digest,
        "bytes": int(path.stat().st_size),
        "rows": int(sidecar["rows"]),
    }
    return frame, sidecar, provenance


def _window_block(window_id, readable, bounds, arrived, closed, slow, still_open):
    first, before = bounds["created_from"], bounds["created_before"]
    days = (date.fromisoformat(before) - date.fromisoformat(first)).days
    assert int(bounds["rows_pulled"]) == arrived, f"{window_id}: sidecar rows_pulled disagrees"
    assert arrived >= closed >= slow and still_open == arrived - closed >= 0, f"{window_id}: counts do not nest"
    return {
        "window_id": window_id,
        "label": readable,
        "created_from": first,
        "created_before": before,
        "days": days,
        "arrived": arrived,
        "closed": closed,
        "still_open_no_label": still_open,
        "closed_later_than_project_threshold": slow,
        "share_of_closed_left_too_long_pct": round(100.0 * slow / closed, 1),
        "denominator": f"closed tickets in {readable.lower()} ({closed:,})",
        "note": "the share is of CLOSED tickets; still-open ones are reported separately and counted neither fast nor slow",
    }


def compare_event_and_quiet():
    """Return the January event-week vs quiet-week comparison as a JSON-ready dict."""
    features, features_sidecar, features_src = _read_verified(FEATURES_PATH)
    labels, labels_sidecar, labels_src = _read_verified(LABELS_PATH)

    merged = labels.merge(features, on="unique_key", how="inner", validate="one_to_one")
    assert len(merged) == len(labels), "a labelled ticket is missing from the features file"

    thresholds = merged["label_threshold_hours"].unique()
    assert len(thresholds) == 1, f"more than one project threshold in this file: {list(thresholds)}"
    project_threshold_hours = float(thresholds[0])
    assert abs(project_threshold_hours - float(labels_sidecar["label_threshold_hours"])) < 1e-9
    assert not OUTCOME_COLUMNS & set(features.columns), "the intake file carries outcome columns"

    arrived = features.groupby("window")["unique_key"].count()
    closed = merged.groupby("window")["unique_key"].count()
    slow = merged.assign(_s=merged["slow_resolution"].astype("int64")).groupby("window")["_s"].sum()
    assert set(arrived.index) == {EVENT, QUIET}, f"unexpected windows: {sorted(arrived.index)}"

    event_block = _window_block(EVENT, "Event week", features_sidecar["windows"][EVENT],
                                int(arrived[EVENT]), int(closed[EVENT]),
                                int(slow[EVENT]), int(arrived[EVENT] - closed[EVENT]))
    quiet_block = _window_block(QUIET, "Quiet week", features_sidecar["windows"][QUIET],
                                int(arrived[QUIET]), int(closed[QUIET]),
                                int(slow[QUIET]), int(arrived[QUIET] - closed[QUIET]))

    arrived_total = sum(int(v) for v in arrived)
    closed_total = int(len(merged))
    slow_total = int(slow.sum())
    pooled = {
        "arrived": arrived_total,
        "closed": closed_total,
        "still_open_no_label": arrived_total - closed_total,
        "closed_later_than_project_threshold": slow_total,
        "share_of_closed_left_too_long_pct": round(100.0 * slow_total / closed_total, 1),
        "method": "sum(slow) / sum(closed) over both windows - not the mean of the two window shares",
        "denominator": f"closed tickets in both windows ({closed_total:,})",
    }

    recorded = labels_sidecar["slow_by_window_pct"]
    assert abs(event_block["share_of_closed_left_too_long_pct"] - float(recorded["event"])) < 0.05
    assert abs(quiet_block["share_of_closed_left_too_long_pct"] - float(recorded["quiet"])) < 0.05
    assert abs(pooled["share_of_closed_left_too_long_pct"]
               - float(labels_sidecar["slow_overall_pct"])) < 0.05
    assert pooled["still_open_no_label"] == int(labels_sidecar["open_tickets"]["count"])
    assert arrived_total == int(features_sidecar["rows"]) and closed_total == int(labels_sidecar["rows"])

    return {
        "report": "heat_jan2024_event_week_vs_quiet_week",
        "kind": "historical comparison of closed tickets - not a model and not a prediction",
        "population": features_sidecar["complaint_filter"],
        "project_threshold": {
            "value_hours": project_threshold_hours,
            "displayed": f"{round(project_threshold_hours, 1)} hours",
            "status": "project threshold - not an SLA and not a published service standard",
            "source": labels_sidecar["threshold_source"],
            "fitted_on": labels_sidecar["threshold_fitted_on"],
            "label_rule": labels_sidecar["label_rule"],
        },
        "windows": {"event_week": event_block, "quiet_week": quiet_block},
        "pooled": pooled,
        "open_tickets": {
            "count": pooled["still_open_no_label"],
            "by_window": {"event_week": event_block["still_open_no_label"],
                          "quiet_week": quiet_block["still_open_no_label"]},
            "rule": labels_sidecar["open_tickets"]["label"],
        },
        "cautions": [
            "the event week is defined by heat intake volume in these dates; no weather field exists in these artefacts, so nothing here attributes the change to weather",
            "the quiet-week share is not independent evidence: the project threshold was fitted on that window's own closed tickets",
            "these are shares of CLOSED tickets; the still-open ones are absent from every denominator",
            "this report describes two January windows and answers no question about future tickets",
        ],
        "sidecar_known_limits": labels_sidecar["known_limits"],
        "sources": {"features_at_intake": features_src, "slow_resolution_labels": labels_src},
    }


PAYLOAD = compare_event_and_quiet()
ROUND_TRIP = json.loads(json.dumps(PAYLOAD, ensure_ascii=False, indent=2))
assert ROUND_TRIP == PAYLOAD, "the dict is not JSON-clean - something survived the round-trip changed"

print(json.dumps(PAYLOAD, ensure_ascii=False, indent=2))
print("\nread back out of the JSON round-trip, not out of pandas:\n")
for key, block in ROUND_TRIP["windows"].items():
    print(f"  {key:11} arrived {block['arrived']:>6,}   closed {block['closed']:>6,}   "
          f"open {block['still_open_no_label']:>3}   slow {block['closed_later_than_project_threshold']:>5,}   "
          f"{block['share_of_closed_left_too_long_pct']:>5.1f}% of closed")
pool = ROUND_TRIP["pooled"]
print(f"  {'pooled':11} arrived {pool['arrived']:>6,}   closed {pool['closed']:>6,}   "
      f"open {pool['still_open_no_label']:>3}   slow {pool['closed_later_than_project_threshold']:>5,}   "
      f"{pool['share_of_closed_left_too_long_pct']:>5.1f}% of closed")
print(f"\nJSON-clean: {len(json.dumps(ROUND_TRIP, ensure_ascii=False)):,} characters, "
      f"top-level keys: {', '.join(ROUND_TRIP)}")

{
  "report": "heat_jan2024_event_week_vs_quiet_week",
  "kind": "historical comparison of closed tickets - not a model and not a prediction",
  "population": "lower(complaint_type) = 'heat/hot water'",
  "project_threshold": {
    "value_hours": 62.34933333333333,
    "displayed": "62.3 hours",
    "status": "project threshold - not an SLA and not a published service standard",
    "source": "p90 of the quiet window's closed tickets pulled in this run, not a published standard",
    "fitted_on": {
      "window": "quiet_2024-01-24_30",
      "n_closed": 7309
    },
    "label_rule": "slow_resolution = resolution_hours > label_threshold_hours, closed tickets only"
  },
  "windows": {
    "event_week": {
      "window_id": "event_2024-01-15_22",
      "label": "Event week",
      "created_from": "2024-01-15",
      "created_before": "2024-01-23",
      "days": 8,
      "arrived": 22811,
      "closed": 22770,
      "still_open_no_label": 41,
      "closed_later_than_project_threshold": 

In [5]:
from resolveiq.heat_weeks import compare_event_and_quiet

report = compare_event_and_quiet()

print(f"{'':11}{'arrived':>10}{'closed':>10}{'open':>7}{'slow':>8}{'share':>9}")
for key in ("event_week", "quiet_week"):
    b = report["windows"][key]
    print(f"{key:11}{b['arrived']:>10,}{b['closed']:>10,}{b['still_open_no_label']:>7}"
          f"{b['closed_later_than_project_threshold']:>8,}"
          f"{b['share_of_closed_left_too_long_pct']:>8.1f}%")
p = report["pooled"]
print(f"{'pooled':11}{p['arrived']:>10,}{p['closed']:>10,}{p['still_open_no_label']:>7}"
      f"{p['closed_later_than_project_threshold']:>8,}{p['share_of_closed_left_too_long_pct']:>8.1f}%")
print()
print("threshold:", report["project_threshold"]["displayed"], "|", report["project_threshold"]["status"])
print("open-ticket rule:", report["open_tickets"]["rule"])
print("source file :", report["sources"]["features_at_intake"]["file"])
print("source hash :", report["sources"]["features_at_intake"]["sha256"][:16], "rows",
      report["sources"]["features_at_intake"]["rows"])

              arrived    closed   open    slow    share
event_week     22,811    22,770     41   6,526    28.7%
quiet_week      7,315     7,309      6     731    10.0%
pooled         30,126    30,079     47   7,257    24.1%

threshold: 62.3 hours | project threshold - not an SLA and not a published service standard
open-ticket rule: none — absent from this file, never labelled False
source file : data/labelled/heat_jan2024_event_quiet_features_at_intake.json
source hash : 8d26ccbb40f73d14 rows 30126


In [6]:
for c in report["cautions"]:
    print("-", c)
print()
print(report["pooled"]["method"])
print(report["pooled"]["denominator"])

- the event week is defined by heat intake volume in these dates; no weather field exists in these artefacts, so nothing here attributes the change to weather
- the quiet-week share is not independent evidence: the project threshold was fitted on that window's own closed tickets
- these are shares of CLOSED tickets; the still-open ones are absent from every denominator
- this report describes two January windows and answers no question about future tickets

sum(slow) / sum(closed) over both windows - not the mean of the two window shares
closed tickets in both windows (30,079)
